In [22]:
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

# Regression Models
from sklearn.linear_model import LinearRegression
# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [23]:
# Setup path
BASE_PATH = 'dataset'
OUTPUT_FOLDER = os.path.join(BASE_PATH, 'output_pipeline')
ARTIFACTS_FOLDER = os.path.join(BASE_PATH, 'artifacts')

In [24]:
# Load fitur (X)
X_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_train.parquet'))
X_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_val.parquet'))
X_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_test.parquet'))

In [25]:
# Load target (y) - convert ke Series untuk sklearn
y_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_train.parquet')).iloc[:, 0]
y_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_val.parquet')).iloc[:, 0]
y_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_test.parquet')).iloc[:, 0]

In [26]:
# Load metadata ID (untuk traceback hasil prediksi nanti)
ID_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_train.parquet'))
ID_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_val.parquet'))
ID_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_test.parquet'))

In [27]:
print(ID_test.head())
print(ID_test.columns)

  kabupaten_kota_asli  tahun
0       Kab. Balangan   2024
1       Kab. Balangan   2025
2     Kota Balikpapan   2024
3     Kota Balikpapan   2025
4         Kab. Banjar   2024
Index(['kabupaten_kota_asli', 'tahun'], dtype='object')


In [28]:
# Load metadata kolom (referensi)   
kolom_info = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'kolom_info.joblib'))

In [29]:
# Verifikasi shape
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}, ID_train: {ID_train.shape}")
print(f"X_val : {X_val.shape}, y_val : {y_val.shape}, ID_val : {ID_val.shape}")
print(f"X_test : {X_test.shape}, y_test : {y_test.shape}, ID_test : {ID_test.shape}")
print(f"\nTotal baris: {len(X_train) + len(X_val) + len(X_test)}")
print(f"Total fitur: {X_train.shape[1]} kolom")
print(f"Target : produktivitas padi (Qu/Ha)")
print(f"Range target train: {y_train.min():.2f} - {y_train.max():.2f}")

X_train: (278, 111), y_train: (278,), ID_train: (278, 2)
X_val : (56, 111), y_val : (56,), ID_val : (56, 2)
X_test : (111, 111), y_test : (111,), ID_test : (111, 2)

Total baris: 445
Total fitur: 111 kolom
Target : produktivitas padi (Qu/Ha)
Range target train: 15.78 - 54.46


In [30]:
# Setup Output Folder
MODEL_FOLDER = os.path.join(BASE_PATH, 'models')
RESULT_FOLDER = os.path.join(BASE_PATH, 'results')

os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(RESULT_FOLDER, exist_ok=True)

In [31]:
print("Mulai training Linear Regression")

# inisialisasi model
lr_model = LinearRegression()

# training model
lr_model.fit(X_train, y_train)

print("Training selesai")

Mulai training Linear Regression
Training selesai


In [32]:
# Prediksi
train_pred = lr_model.predict(X_train)
val_pred = lr_model.predict(X_val)
test_pred = lr_model.predict(X_test)

In [33]:
def hitung_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [34]:
# evaluasi train
train_mae, train_rmse, train_r2 = hitung_metrics(
    y_train,
    train_pred
)

In [35]:
# evaluasi validation
val_mae, val_rmse, val_r2 = hitung_metrics(
    y_val,
    val_pred
)

In [36]:
# evaluasi test
test_mae, test_rmse, test_r2 = hitung_metrics(
    y_test,
    test_pred
)

In [37]:
print("HASIL LINEAR REGRESSION")

print("\nTrain")
print(f"MAE  : {train_mae:.4f}")
print(f"RMSE : {train_rmse:.4f}")
print(f"R2   : {train_r2:.4f}")

print("\nValidation")
print(f"MAE  : {val_mae:.4f}")
print(f"RMSE : {val_rmse:.4f}")
print(f"R2   : {val_r2:.4f}")

print("\nTest")
print(f"MAE  : {test_mae:.4f}")
print(f"RMSE : {test_rmse:.4f}")
print(f"R2   : {test_r2:.4f}")

HASIL LINEAR REGRESSION

Train
MAE  : 2.1307
RMSE : 2.8125
R2   : 0.8290

Validation
MAE  : 3.7149
RMSE : 4.7957
R2   : 0.4651

Test
MAE  : 5.2168
RMSE : 6.3330
R2   : 0.2516


In [38]:
# Simpan Model Linear Regression
joblib.dump(
    lr_model,
    os.path.join(BASE_PATH, 'models', 'linear_regression.joblib')
)
print("Model berhasil disimpan.")

Model berhasil disimpan.


In [39]:
# Simpan metrics
metrics_lr = pd.DataFrame({
    "model": ["Linear Regression"],
    "train_mae": [train_mae],
    "train_rmse": [train_rmse],
    "train_r2": [train_r2],
    "val_mae": [val_mae],
    "val_rmse": [val_rmse],
    "val_r2": [val_r2],
    "test_mae": [test_mae],
    "test_rmse": [test_rmse],
    "test_r2": [test_r2]
})

os.makedirs("results", exist_ok=True)

metrics_lr.to_csv(
    os.path.join(BASE_PATH, "results", "linear_regression_metrics.csv"),
    index=False
)
print("Metrics berhasil disimpan.")

Metrics berhasil disimpan.


In [ ]:
# Simpan hasil prediksi test
hasil_pred_lr = ID_test.copy()

hasil_pred_lr["actual"] = y_test.values
hasil_pred_lr["prediction_lr"] = test_pred

hasil_pred_lr.to_csv(
    os.path.join(BASE_PATH, "results", "linear_regression_predictions.csv"),
    index=False
)

print("Hasil prediksi berhasil disimpan.")

# preview hasil
print("\nPreview prediction:")
print(hasil_pred_lr.head())

Hasil prediksi berhasil disimpan.

Preview prediction:
  kabupaten_kota_asli  tahun  actual  prediction_lr
0       Kab. Balangan   2024   36.24      45.867961
1       Kab. Balangan   2025   38.35      43.727954
2     Kota Balikpapan   2024   25.34      36.587881
3     Kota Balikpapan   2025   39.36      36.995476
4         Kab. Banjar   2024   37.79      41.116102
